In [ ]:
%cd /content

!test -d /content/CS_769_Project || git clone https://github.com/Achintya2510/CS_769_Project.git

%cd /content/CS_769_Project
!git pull --ff-only origin main
%pip install -q -r requirements.txt

/content
Cloning into 'CS_769_Project'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 27 (delta 10), reused 24 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 39.61 KiB | 2.48 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/CS_769_Project
From https://github.com/Achintya2510/CS_769_Project
 * branch            main       -> FETCH_HEAD
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.27.0 re

# Colab GPU workflow
This notebook contains only GPU-heavy stages. Data auditing, general preprocessing, plots, and report writing stay local. All expensive cells are disabled by default.

In [ ]:
import torch
import transformers
import sentence_transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 4.57.6
Sentence Transformers: 5.7.0
GPU available: True
GPU: Tesla T4


In [ ]:
import importlib
import json
import random
from pathlib import Path

import torch
import training

importlib.reload(training)

from data_pipeline import load_config, load_jsonl
from models import (
    FixedAnswerer,
    FrozenTextEncoder,
    build_actor_critic,
    create_embedding_cache,
    load_embedding_cache,
)
from training import (
    build_environment_factory,
    export_inference_bundle,
    mean_greedy_evidence_f1,
    mean_greedy_episode_reward,
    set_seed,
    train_supervised,
    train_ppo,
)

config = load_config("/content/CS_769_Project/config.yaml")

drive_root = Path("/content/drive/MyDrive/CS_769_Project")
config["paths"]["processed_dir"] = str(drive_root / "data" / "processed")
config["paths"]["output_dir"] = str(drive_root / "outputs")

device = torch.device("cuda")
print("Updated code loaded")
print("GPU:", torch.cuda.get_device_name(0))
print("Supervised batch size:", config["training"]["supervised_batch_size"])
print("PPO minibatch size:", config["training"]["ppo_minibatch_size"])

Updated code loaded
GPU: Tesla T4
Supervised batch size: 32
PPO minibatch size: 128


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import torch

from data_pipeline import load_config, load_jsonl
from models import FixedAnswerer, FrozenTextEncoder, build_actor_critic, create_embedding_cache, load_embedding_cache
from training import (build_environment_factory, export_inference_bundle, mean_greedy_evidence_f1,
                      mean_greedy_episode_reward, set_seed, train_supervised, train_ppo)

assert torch.cuda.is_available(), 'Select a GPU runtime in Colab first.'
config = load_config('config.yaml')
drive_root = Path('/content/drive/MyDrive/CS_769_Project')
config['paths']['processed_dir'] = str(drive_root / 'data' / 'processed')
config['paths']['output_dir'] = str(drive_root / 'outputs')
device = torch.device('cuda')
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
Tesla T4


In [ ]:
from pathlib import Path
import torch

output_dir = Path(config["paths"]["output_dir"])
embedding_dir = output_dir / "embeddings"
runs_dir = output_dir / "colab_runs"

required_files = [
    embedding_dir / "evidence_embeddings.npy",
    embedding_dir / "evidence_index.json",
    embedding_dir / "question_embeddings.npy",
    embedding_dir / "question_index.json",
]

for seed in config["training"]["seeds"]:
    required_files.append(
        runs_dir / f"supervised_seed_{seed}" / "best_model.pt"
    )

missing = [path for path in required_files if not path.exists()]

if missing:
    print("Missing required files:")
    for path in missing:
        print(path)
    raise FileNotFoundError("Required PPO files are missing.")

print("All embeddings and supervised models found.")
print("GPU:", torch.cuda.get_device_name(0))

# Show all resumable PPO runs
checkpoints = sorted(runs_dir.glob("ppo_*/latest.pt"))

if not checkpoints:
    print("No PPO checkpoint found: PPO will begin from episode 0.")
else:
    print("\nExisting PPO checkpoints:")
    for checkpoint_path in checkpoints:
        state = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
        episode = state["progress"]["episode"]
        print(
            checkpoint_path.parent.name,
            f"episode {episode}/{config['training']['max_episodes']}",
        )
        del state

reward_cache = output_dir / "answer_reward_cache.json"
print("\nAnswer-reward cache:", reward_cache.exists())

All embeddings and supervised models found.
GPU: Tesla T4

Existing PPO checkpoints:
ppo_answer_only_seed_13 episode 8192/50000

Answer-reward cache: True


## 1. Full embedding cache

In [ ]:
RUN_EMBEDDINGS = False
embedding_dir = Path(config['paths']['output_dir']) / 'embeddings'
if RUN_EMBEDDINGS:
    encoder = FrozenTextEncoder(
        config['models']['evidence_encoder'],
        config['models']['evidence_encoder_revision'],
        device='cuda',
    )
    counts = create_embedding_cache(encoder, config['paths']['processed_dir'], embedding_dir)
    print(counts)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9521 [00:00<?, ?it/s]

Batches:   0%|          | 0/1034 [00:00<?, ?it/s]

{'evidence': 609286, 'questions': 66148}


## 2. Supervised selector training

In [ ]:
RUN_SUPERVISED = False

if RUN_SUPERVISED:
    evidence_embeddings, question_embeddings = load_embedding_cache(
        Path(config["paths"]["output_dir"]) / "embeddings"
    )

    train_rows = load_jsonl(
        Path(config["paths"]["processed_dir"])
        / "questions"
        / "train_core.jsonl"
    )

    val_rows = load_jsonl(
        Path(config["paths"]["processed_dir"])
        / "questions"
        / "train_val.jsonl"
    )

    question_ids = [
        row["question_id"]
        for row in train_rows
    ]

    validation_count = min(
        config["training"]["validation_examples"],
        len(val_rows),
    )

    validation_rng = random.Random(
        config["data"]["split_seed"]
    )

    val_ids = validation_rng.sample(
        [row["question_id"] for row in val_rows],
        validation_count,
    )

    env_factory, _ = build_environment_factory(
        train_rows,
        config["paths"]["processed_dir"],
        evidence_embeddings,
        question_embeddings,
        config,
        reward_variant="evidence_only",
    )

    val_factory, _ = build_environment_factory(
        val_rows,
        config["paths"]["processed_dir"],
        evidence_embeddings,
        question_embeddings,
        config,
        reward_variant="evidence_only",
    )

    for seed in config["training"]["seeds"]:
        set_seed(seed)

        model = build_actor_critic(config).to(device)

        run_dir = (
            Path(config["paths"]["output_dir"])
            / "colab_runs"
            / f"supervised_seed_{seed}"
        )

        history = train_supervised(
            model,
            env_factory,
            question_ids,
            epochs=config["training"]["supervised_epochs"],
            learning_rate=config["training"]["supervised_learning_rate"],
            device=device,
            validation_fn=lambda current: mean_greedy_evidence_f1(
                current,
                val_factory,
                val_ids,
                device,
            ),
            batch_size=config["training"]["supervised_batch_size"],
            output_dir=run_dir,
        )

        run_dir.mkdir(parents=True, exist_ok=True)

        torch.save(
            model.state_dict(),
            run_dir / "best_model.pt",
        )

        (run_dir / "history.json").write_text(
            json.dumps(history, indent=2)
        )

        print(
            f"Completed seed {seed}; "
            f"checkpoint saved to {run_dir}"
        )

supervised epoch 1/3:   0%|          | 0/56476 [00:00<?, ?it/s]

/content/CS_769_Project/training.py:226: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  question = torch.as_tensor(env.question_embedding, dtype=torch.float32)


supervised epoch 2/3:   0%|          | 0/56476 [00:00<?, ?it/s]

RuntimeError: cudnn RNN backward can only be called in training mode

In [ ]:
for seed in config["training"]["seeds"]:
    history_path = (
        Path(config["paths"]["output_dir"])
        / "colab_runs"
        / f"supervised_seed_{seed}"
        / "history.json"
    )

    history = json.loads(history_path.read_text())

    print(f"\nSeed {seed}")
    for row in history:
        print(row)


Seed 13
{'epoch': 1.0, 'loss': 1.0656455804898897, 'validation_evidence_f1': 0.3296296296296296}
{'epoch': 2.0, 'loss': 0.977684275364272, 'validation_evidence_f1': 0.3269360269360269}
{'epoch': 3.0, 'loss': 0.918489339333289, 'validation_evidence_f1': 0.3484848484848485}

Seed 42
{'epoch': 1.0, 'loss': 1.0665707768932828, 'validation_evidence_f1': 0.32222222222222224}
{'epoch': 2.0, 'loss': 0.9779784599159939, 'validation_evidence_f1': 0.33468013468013463}
{'epoch': 3.0, 'loss': 0.9230329959103234, 'validation_evidence_f1': 0.3478114478114478}

Seed 2026
{'epoch': 1.0, 'loss': 1.0650891800682747, 'validation_evidence_f1': 0.3296296296296296}
{'epoch': 2.0, 'loss': 0.9776105726054855, 'validation_evidence_f1': 0.32525252525252524}
{'epoch': 3.0, 'loss': 0.9231051316580631, 'validation_evidence_f1': 0.3303030303030303}


In [ ]:
from pathlib import Path

local_root = Path("/content/ppo_runtime")
(local_root / "processed").mkdir(parents=True, exist_ok=True)
(local_root / "embeddings").mkdir(parents=True, exist_ok=True)

print("Copying processed data to Colab local storage...")
!rsync -a --info=progress2 \
    "/content/drive/MyDrive/CS_769_Project/data/processed/" \
    "/content/ppo_runtime/processed/"

print("Copying embeddings to Colab local storage...")
!rsync -a --info=progress2 \
    "/content/drive/MyDrive/CS_769_Project/outputs/embeddings/" \
    "/content/ppo_runtime/embeddings/"

config["paths"]["processed_dir"] = "/content/ppo_runtime/processed"
embedding_dir = Path("/content/ppo_runtime/embeddings")

print("Local processed data:", Path(config["paths"]["processed_dir"]).exists())
print("Local embeddings:", embedding_dir.exists())
print("Checkpoint output remains:", config["paths"]["output_dir"])

Copying processed data to Colab local storage...
    647,775,970 100%    7.53MB/s    0:01:22 (xfr#12382, to-chk=0/12385)
Copying embeddings to Colab local storage...
  1,086,533,810 100%   66.58MB/s    0:00:15 (xfr#4, to-chk=0/5)
Local processed data: True
Local embeddings: True
Checkpoint output remains: /content/drive/MyDrive/CS_769_Project/outputs


## 3. PPO pilot and full runs
Do not enable until preprocessing, embeddings, baselines, supervised checkpoints, and local PPO sanity checks are complete.

In [ ]:
RUN_PPO = True
PPO_VARIANTS = ['answer_only', 'evidence_only', 'combined', 'combined_step']
if RUN_PPO:
    from evaluation import AnswerRewardCache
    evidence_embeddings, question_embeddings = load_embedding_cache(embedding_dir)
    train_rows = load_jsonl(Path(config['paths']['processed_dir']) / 'questions' / 'train_core.jsonl')
    val_rows = load_jsonl(Path(config['paths']['processed_dir']) / 'questions' / 'train_val.jsonl')
    examples_by_id = {row['question_id']: row for row in train_rows + val_rows}
    question_ids = [row['question_id'] for row in train_rows]
    _, train_evidence = build_environment_factory(
        train_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only'
    )
    _, val_evidence = build_environment_factory(
        val_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only'
    )
    evidence_by_id = {**train_evidence, **val_evidence}
    answerer = FixedAnswerer(
        config['models']['answer_model'], config['models']['answer_model_revision'],
        device='cuda', max_input_tokens=config['models']['max_input_tokens'],
        max_output_tokens=config['models']['max_output_tokens']
    )
    answer_reward = AnswerRewardCache(
        Path(config['paths']['output_dir']) / 'answer_reward_cache.json', answerer,
        examples_by_id, evidence_by_id, config['models']['answer_model']
    )
    for variant in PPO_VARIANTS:
        for seed in config['training']['seeds']:
            set_seed(seed)
            model = build_actor_critic(config).to(device)
            supervised_path = Path(config['paths']['output_dir']) / 'colab_runs' / f'supervised_seed_{seed}' / 'best_model.pt'
            model.load_state_dict(torch.load(supervised_path, map_location=device, weights_only=True))
            reward_callback = None if variant == 'evidence_only' else answer_reward
            env_factory, _ = build_environment_factory(
                train_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
                config, answer_reward_fn=reward_callback, reward_variant=variant
            )
            val_factory, _ = build_environment_factory(
                val_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
                config, answer_reward_fn=reward_callback, reward_variant=variant
            )
            run_dir = Path(config['paths']['output_dir']) / 'colab_runs' / f'ppo_{variant}_seed_{seed}'
            validation_ids = [row['question_id'] for row in val_rows[:500]]
            train_ppo(
                model, env_factory, question_ids, config, run_dir, device,
                validation_fn=lambda current: mean_greedy_episode_reward(
                    current, val_factory, validation_ids, device
                ),
            )

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

PPO episodes:  16%|#6        | 8192/50000 [00:00<?, ?it/s]

PPO episodes:   0%|          | 0/50000 [00:00<?, ?it/s]

## 4. Final answer generation and GPU timing
Run only after the checkpoint is selected using `train_val`.

In [ ]:
RUN_FINAL_EVALUATION = False
FINAL_VARIANT = 'combined_step'
FINAL_SEED = 42
if RUN_FINAL_EVALUATION:
    from evaluation import (generate_answers, prepare_direct_inputs, prepare_policy_inputs,
                            prepare_similarity_inputs, save_jsonl)

    evidence_embeddings, question_embeddings = load_embedding_cache(embedding_dir)
    dev_rows = load_jsonl(Path(config['paths']['processed_dir']) / 'questions' / 'dev.jsonl')
    dev_factory, evidence_by_id = build_environment_factory(
        dev_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only'
    )
    answerer = FixedAnswerer(
        config['models']['answer_model'], config['models']['answer_model_revision'],
        device='cuda', max_input_tokens=config['models']['max_input_tokens'],
        max_output_tokens=config['models']['max_output_tokens']
    )
    result_dir = Path(config['paths']['output_dir']) / 'results'
    baseline_inputs = {
        'direct': prepare_direct_inputs(dev_rows, config['paths']['processed_dir']),
        'similarity': prepare_similarity_inputs(dev_rows, dev_factory, evidence_by_id),
    }
    for name, inputs in baseline_inputs.items():
        save_jsonl(result_dir / f'{name}.jsonl', generate_answers(answerer, inputs, batch_size=16))
    run_specs = [('supervised', None, seed) for seed in config['training']['seeds']]
    run_specs += [('ppo', variant, seed) for variant in PPO_VARIANTS for seed in config['training']['seeds']]
    for kind, variant, seed in run_specs:
        model = build_actor_critic(config).to(device)
        if kind == 'supervised':
            path = Path(config['paths']['output_dir']) / 'colab_runs' / f'supervised_seed_{seed}' / 'best_model.pt'
            state = torch.load(path, map_location=device, weights_only=True)
            name = f'supervised_seed_{seed}'
        else:
            path = Path(config['paths']['output_dir']) / 'colab_runs' / f'ppo_{variant}_seed_{seed}' / 'best.pt'
            state = torch.load(path, map_location=device, weights_only=False)['model']
            name = f'ppo_{variant}_seed_{seed}'
        model.load_state_dict(state)
        inputs = prepare_policy_inputs(model, dev_rows, dev_factory, evidence_by_id, device)
        save_jsonl(result_dir / f'{name}.jsonl', generate_answers(answerer, inputs, batch_size=16))
    print(result_dir)

## 5. Export final local inference bundle

In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    import numpy as np
    final_model = build_actor_critic(config).to(device)
    checkpoint_path = Path(config['paths']['output_dir']) / 'colab_runs' / f'ppo_{FINAL_VARIANT}_seed_{FINAL_SEED}' / 'best.pt'
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    final_model.load_state_dict(checkpoint['model'])
    evidence_matrix = np.load(embedding_dir / 'evidence_embeddings.npy')
    evidence_index = json.loads((embedding_dir / 'evidence_index.json').read_text())
    preprocessing_manifest = json.loads(
        (Path(config['paths']['processed_dir']) / 'preprocessing_manifest.json').read_text()
    )
    bundle_dir = Path(config['paths']['output_dir']) / 'final_bundle'
    manifest = export_inference_bundle(
        bundle_dir, final_model, config, evidence_matrix, evidence_index, preprocessing_manifest
    )
    print(json.dumps(manifest, indent=2)[:3000])